# OULAD Missing-Value Investigation

## Purpose

This notebook explains the three monitored missing-value conditions found in the OULAD source:

- 1,111 missing `imd_band` values
- 45 missing `date_registration` values
- 173 missing `score` values

The goal is to show which rows are affected, confirm whether Bronze or Silver created the issue, explain what the data supports, and document how downstream models should handle each value. The counts written in the notes below describe the attached OULAD snapshot. Run the SQL cells again whenever the source data changes.

## How the values move through the pipeline

The CSV files use `?` for an unknown value. Bronze standardizes `?` and empty text to SQL `NULL`. Silver validates the row and keeps the `NULL`; it does not invent a replacement.

`CSV ?` → `Bronze NULL` → `Silver NULL` → `DQ WARNING` → `Gold and Analytics apply a documented null rule`

This is a transformation, not data loss. `NULL` is the database's standard way to represent an unknown value. It can be tested, counted, and handled consistently.

In [0]:
%sql
-- One-screen overview of the three monitored issues.
WITH issue_counts AS (
  SELECT
    'Missing IMD band' AS issue,
    COUNT(*) AS total_rows,
    COUNT_IF(imd_band IS NULL) AS affected_rows,
    5.0 AS allowed_pct
  FROM `ftw-week-07`.`02-clean`.student_info_clean

  UNION ALL

  SELECT
    'Missing registration date',
    COUNT(*),
    COUNT_IF(date_registration IS NULL),
    1.0
  FROM `ftw-week-07`.`02-clean`.student_registration_clean

  UNION ALL

  SELECT
    'Missing assessment score',
    COUNT(*),
    COUNT_IF(score IS NULL),
    1.0
  FROM `ftw-week-07`.`02-clean`.student_assessment_clean
)
SELECT
  issue,
  total_rows,
  affected_rows,
  ROUND(100.0 * affected_rows / NULLIF(total_rows, 0), 3) AS missing_pct,
  allowed_pct,
  CASE
    WHEN affected_rows = 0 THEN 'PASS'
    WHEN 100.0 * affected_rows / NULLIF(total_rows, 0) <= allowed_pct THEN 'WARNING'
    ELSE 'FAIL'
  END AS status
FROM issue_counts
ORDER BY issue;

## Expected overview for this snapshot

| Check | Missing rows | Total rows | Missing rate | Limit | Expected status |
|---|---:|---:|---:|---:|---|
| IMD band | 1,111 | 32,593 | 3.409% | 5% | WARNING |
| Registration date | 45 | 32,593 | 0.138% | 1% | WARNING |
| Assessment score | 173 | 173,912 | 0.099% | 1% | WARNING |

A warning means the issue exists but remains within the accepted monitoring limit. The quality gate stops the pipeline only when a `CRITICAL` check fails. These three completeness checks have `MEDIUM` severity, so they remain visible without blocking valid data from moving forward.

In [0]:
%sql
-- Prove that Silver did not create, remove, or fill these missing values.
SELECT
  'IMD band' AS issue,
  (SELECT COUNT(*) FROM `ftw-week-07`.`01-raw`.student_info) AS bronze_rows,
  (SELECT COUNT(*) FROM `ftw-week-07`.`02-clean`.student_info_clean) AS silver_rows,
  (SELECT COUNT_IF(imd_band IS NULL) FROM `ftw-week-07`.`01-raw`.student_info) AS bronze_missing,
  (SELECT COUNT_IF(imd_band IS NULL) FROM `ftw-week-07`.`02-clean`.student_info_clean) AS silver_missing

UNION ALL

SELECT
  'Registration date',
  (SELECT COUNT(*) FROM `ftw-week-07`.`01-raw`.student_registration),
  (SELECT COUNT(*) FROM `ftw-week-07`.`02-clean`.student_registration_clean),
  (SELECT COUNT_IF(date_registration IS NULL) FROM `ftw-week-07`.`01-raw`.student_registration),
  (SELECT COUNT_IF(date_registration IS NULL) FROM `ftw-week-07`.`02-clean`.student_registration_clean)

UNION ALL

SELECT
  'Assessment score',
  (SELECT COUNT(*) FROM `ftw-week-07`.`01-raw`.student_assessment),
  (SELECT COUNT(*) FROM `ftw-week-07`.`02-clean`.student_assessment_clean),
  (SELECT COUNT_IF(score IS NULL) FROM `ftw-week-07`.`01-raw`.student_assessment),
  (SELECT COUNT_IF(score IS NULL) FROM `ftw-week-07`.`02-clean`.student_assessment_clean);

### How to explain the comparison

The Bronze and Silver row counts should match for these three tables, and the missing counts should also match. This proves the missing values came from the source CSV and were not produced by the cleaning query. Silver changed the representation from `?` to `NULL`, but preserved the affected rows.

# 1. Missing IMD band

`imd_band` is a socioeconomic deprivation category. It is descriptive information, not a measure that can be safely calculated from grades, location, or final result.

In [0]:
%sql
-- Exact 1,111 affected enrollment rows.
SELECT
  student.code_module,
  student.code_presentation,
  student.id_student,
  student.gender,
  student.region,
  student.highest_education,
  student.imd_band,
  student.age_band,
  student.disability,
  student.final_result,
  registration.date_registration,
  registration.date_unregistration
FROM `ftw-week-07`.`02-clean`.student_info_clean AS student
LEFT JOIN `ftw-week-07`.`02-clean`.student_registration_clean AS registration
  ON student.code_module = registration.code_module
  AND student.code_presentation = registration.code_presentation
  AND student.id_student = registration.id_student
WHERE student.imd_band IS NULL
ORDER BY student.code_module, student.code_presentation, student.id_student;

In [0]:
%sql
-- Profile the affected enrollment rows without guessing the missing band.
WITH affected AS (
  SELECT *
  FROM `ftw-week-07`.`02-clean`.student_info_clean
  WHERE imd_band IS NULL
)
SELECT
  final_result,
  COUNT(*) AS affected_rows,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS affected_pct
FROM affected
GROUP BY final_result
ORDER BY affected_rows DESC;

## What I found and what we will do

- 1,111 of 32,593 enrollment rows have no IMD band, or 3.41%.
- The affected rows still have valid student, module, demographic, and final-result information. All 1,111 also match a registration row.
- The affected outcomes include 531 Pass, 236 Withdrawn, 199 Distinction, and 145 Fail. Missing IMD therefore does not represent one outcome group.

We will keep every row and keep `imd_band` as `NULL` in Silver. We will not guess the band from region, education, or performance because that could create a false socioeconomic classification. In Gold, the demographic key uses `UNKNOWN` when building the key so the row remains joinable, while the descriptive `imd_band` value remains null. Dashboards can display `COALESCE(imd_band, 'Unknown')`. Ordered IMD comparisons should exclude the null band and show an `Unknown IMD count` separately.

## Why this shows a warning

The missing rate is 3.41%, below the 5% monitoring limit. The issue is real, so it is not a perfect pass. It is a `MEDIUM` warning and does not stop the pipeline. A verified value should replace the null only if a trusted source later provides it.

# 2. Missing registration date

`date_registration` is the number of days relative to the module-presentation start. A null date means the enrollment exists, but its exact registration timing is unknown.

In [0]:
%sql
-- Exact 45 affected registration rows.
SELECT
  registration.code_module,
  registration.code_presentation,
  registration.id_student,
  registration.date_registration,
  registration.date_unregistration,
  student.gender,
  student.region,
  student.imd_band,
  student.final_result
FROM `ftw-week-07`.`02-clean`.student_registration_clean AS registration
INNER JOIN `ftw-week-07`.`02-clean`.student_info_clean AS student
  ON registration.code_module = student.code_module
  AND registration.code_presentation = student.code_presentation
  AND registration.id_student = student.id_student
WHERE registration.date_registration IS NULL
ORDER BY registration.code_module, registration.code_presentation, registration.id_student;

In [0]:
%sql
-- Summarize the evidence available for the 45 affected rows.
WITH affected AS (
  SELECT
    registration.*,
    student.final_result
  FROM `ftw-week-07`.`02-clean`.student_registration_clean AS registration
  INNER JOIN `ftw-week-07`.`02-clean`.student_info_clean AS student
    ON registration.code_module = student.code_module
    AND registration.code_presentation = student.code_presentation
    AND registration.id_student = student.id_student
  WHERE registration.date_registration IS NULL
)
SELECT 'All affected registrations' AS metric, COUNT(*) AS row_count,
  ROUND(100.0 * COUNT(*) / 45, 1) AS pct_of_affected
FROM affected

UNION ALL

SELECT 'Withdrawn', COUNT(*), ROUND(100.0 * COUNT(*) / 45, 1)
FROM affected WHERE final_result = 'Withdrawn'

UNION ALL

SELECT 'Fail', COUNT(*), ROUND(100.0 * COUNT(*) / 45, 1)
FROM affected WHERE final_result = 'Fail'

UNION ALL

SELECT 'Pass', COUNT(*), ROUND(100.0 * COUNT(*) / 45, 1)
FROM affected WHERE final_result = 'Pass'

UNION ALL

SELECT 'Known unregistration date', COUNT(*), ROUND(100.0 * COUNT(*) / 45, 1)
FROM affected WHERE date_unregistration IS NOT NULL;

## What I found and what we will do

- 45 of 32,593 registration rows have no registration date, or 0.14%.
- All 45 rows match valid student enrollment records, so they are not orphan records.
- Their final results are 39 Withdrawn, 5 Fail, and 1 Pass. This means 44 of 45, or 97.8%, withdrew or failed.
- 39 of the 45 rows still have a known unregistration date. That supports keeping the record, but it does not reveal the missing registration date.

We will keep the rows and preserve the unknown registration date as `NULL`. We will never replace it with `0`, because day 0 means the student registered on the course start date. Registration-timing calculations should use `WHERE date_registration IS NOT NULL` and publish an `unknown_registration_date_count` beside the result. A missingness flag can be added for analysis, but it must not be presented as proof that the missing date caused withdrawal.

## Why this shows a warning

The missing rate is 0.14%, below the 1% monitoring limit. The condition is `MEDIUM`, so it is recorded as a warning and the pipeline continues. We can only fill the date if a verified registration source becomes available.

# 3. Missing assessment score

A blank score means the submission has no recorded grade. It does not mean the student received a score of zero.

In [0]:
%sql
-- Exact 173 affected submission rows, including context for investigation.
WITH all_submissions AS (
  SELECT
    submission.id_assessment,
    submission.id_student,
    submission.date_submitted,
    submission.is_banked,
    submission.score,
    assessment.code_module,
    assessment.code_presentation,
    assessment.assessment_type,
    assessment.assessment_date
  FROM `ftw-week-07`.`02-clean`.student_assessment_clean AS submission
  INNER JOIN `ftw-week-07`.`02-clean`.assessments_clean AS assessment
    ON submission.id_assessment = assessment.id_assessment
),
graded_by_enrollment AS (
  SELECT
    code_module,
    code_presentation,
    id_student,
    COUNT_IF(score IS NOT NULL) AS other_graded_submission_count
  FROM all_submissions
  GROUP BY code_module, code_presentation, id_student
)
SELECT
  submission.code_module,
  submission.code_presentation,
  submission.id_student,
  submission.id_assessment,
  submission.assessment_type,
  submission.assessment_date AS due_relative_day,
  submission.date_submitted AS submitted_relative_day,
  submission.date_submitted - submission.assessment_date AS days_late,
  submission.is_banked,
  submission.score,
  student.final_result,
  graded.other_graded_submission_count
FROM all_submissions AS submission
INNER JOIN `ftw-week-07`.`02-clean`.student_info_clean AS student
  ON submission.code_module = student.code_module
  AND submission.code_presentation = student.code_presentation
  AND submission.id_student = student.id_student
INNER JOIN graded_by_enrollment AS graded
  ON submission.code_module = graded.code_module
  AND submission.code_presentation = graded.code_presentation
  AND submission.id_student = graded.id_student
WHERE submission.score IS NULL
ORDER BY submission.code_module, submission.code_presentation, submission.id_student, submission.id_assessment;

In [0]:
%sql
-- Reproduce the presentation findings for blank-score submissions.
WITH all_submissions AS (
  SELECT
    submission.id_assessment,
    submission.id_student,
    submission.date_submitted,
    submission.score,
    assessment.code_module,
    assessment.code_presentation,
    assessment.assessment_type,
    assessment.assessment_date
  FROM `ftw-week-07`.`02-clean`.student_assessment_clean AS submission
  INNER JOIN `ftw-week-07`.`02-clean`.assessments_clean AS assessment
    ON submission.id_assessment = assessment.id_assessment
),
affected_rows AS (
  SELECT
    submission.*,
    student.final_result
  FROM all_submissions AS submission
  INNER JOIN `ftw-week-07`.`02-clean`.student_info_clean AS student
    ON submission.code_module = student.code_module
    AND submission.code_presentation = student.code_presentation
    AND submission.id_student = student.id_student
  WHERE submission.score IS NULL
),
affected_enrollments AS (
  SELECT DISTINCT code_module, code_presentation, id_student, final_result
  FROM affected_rows
),
enrollment_grading AS (
  SELECT
    affected.code_module,
    affected.code_presentation,
    affected.id_student,
    COUNT_IF(all_work.score IS NOT NULL) AS other_graded_count
  FROM affected_enrollments AS affected
  LEFT JOIN all_submissions AS all_work
    ON affected.code_module = all_work.code_module
    AND affected.code_presentation = all_work.code_presentation
    AND affected.id_student = all_work.id_student
  GROUP BY affected.code_module, affected.code_presentation, affected.id_student
)
SELECT 'Blank-score submission rows' AS metric, COUNT(*) AS affected_count, 173 AS denominator,
  ROUND(100.0 * COUNT(*) / 173, 1) AS pct
FROM affected_rows

UNION ALL

SELECT 'TMA blank-score rows', COUNT(*), 173, ROUND(100.0 * COUNT(*) / 173, 1)
FROM affected_rows WHERE assessment_type = 'TMA'

UNION ALL

SELECT 'More than 14 days late', COUNT(*), 173, ROUND(100.0 * COUNT(*) / 173, 1)
FROM affected_rows WHERE date_submitted - assessment_date > 14

UNION ALL

SELECT 'Affected enrollments', COUNT(*), 161, ROUND(100.0 * COUNT(*) / 161, 1)
FROM affected_enrollments

UNION ALL

SELECT 'Affected enrollments that withdrew or failed', COUNT(*), 161,
  ROUND(100.0 * COUNT(*) / 161, 1)
FROM affected_enrollments WHERE final_result IN ('Withdrawn', 'Fail')

UNION ALL

SELECT 'Affected enrollments with other graded work', COUNT(*), 161,
  ROUND(100.0 * COUNT(*) / 161, 1)
FROM enrollment_grading WHERE other_graded_count > 0;

## What I found and what we will do

- 173 of 173,912 submission rows have no score, or 0.10%.
- All 173 blank scores are TMAs, which are tutor-marked assessments.
- The 173 rows represent 161 student enrollments. Of these, 127, or 78.9%, ended in Withdrawn or Fail.
- 81 of the 173 submissions, or 46.8%, were more than 14 days late.
- 138 of the 161 affected enrollments, or 85.7%, still had at least one other graded submission. A blank score therefore does not automatically mean the student was inactive.

These patterns support the interpretation that the null scores are genuine unrecorded or ungraded cases, not a Silver cleaning error. They do not prove the exact reason for every blank score. We will keep the rows, keep `score` as `NULL`, and monitor the count. Average-score and pass-rate calculations must use scored submissions only. Use `WHERE score IS NOT NULL` or an equivalent eligible denominator. Never change a blank score to zero, because zero is an actual grade.

## Why this shows a warning

The missing rate is 0.10%, below the 1% monitoring limit. The check has `MEDIUM` severity, so the pipeline continues. The dashboard should show `missing_score_count` separately from scored submissions.

# Resolution policy

## What we can resolve inside the pipeline

1. Standardize source `?` and blank text to SQL `NULL`.
2. Preserve the complete row when its keys and relationships are valid.
3. Monitor the count and percentage in both Bronze and Silver.
4. Use an `Unknown` display category for missing IMD values.
5. Exclude unknown registration dates from timing calculations and report their count.
6. Exclude null scores from grade averages and pass-rate denominators, while reporting their count.

## What the pipeline cannot resolve by itself

The pipeline cannot reconstruct the true IMD band, registration date, or score from the other columns. Filling these values without a verified source would be an assumption, not cleaning. If a trusted correction arrives later, update the source or apply a documented correction table with the record key, replacement value, source, owner, and correction date.

In [0]:
%sql
-- Show the latest persisted Bronze and Silver results for these checks.
WITH relevant_results AS (
  SELECT
    *,
    DENSE_RANK() OVER (PARTITION BY layer ORDER BY executed_at DESC) AS run_rank
  FROM `ftw-week-07`.`04-analytics`.dq_check_results
  WHERE layer IN ('BRONZE', 'SILVER')
    AND (
      column_name = 'imd_band'
      OR column_name = 'date_registration'
      OR column_name = 'score'
    )
)
SELECT
  layer,
  dataset_name,
  column_name,
  check_name,
  total_count,
  failed_count,
  failure_pct,
  threshold_pct,
  severity,
  status,
  executed_at
FROM relevant_results
WHERE run_rank = 1
ORDER BY layer, dataset_name, column_name;

# Short presentation script

During profiling, we found three real source-level missing-value conditions: 1,111 IMD bands, 45 registration dates, and 173 assessment scores. Bronze converts the source question-mark token to a proper SQL null. Silver validates the rows but keeps them because their keys and relationships are valid.

We did not fill the values with guesses. Missing IMD is displayed as Unknown, missing registration dates are excluded from timing metrics, and missing scores are excluded from grade averages rather than treated as zero. Each issue is counted separately in the data-quality results. Their rates remain within our documented limits, so they appear as Medium warnings. The pipeline stops only when a Critical validation fails.